# 多个角色怎样交接任务

本章学习目标：

- 理解 planner、executor、writer、reviewer、critic 和 router 的职责。
- 用明确流程组织协作，规定每个角色的输入和输出。
- 处理反复修订、任务漂移、无效交接和上下文增长。
- 比较单 Agent 与多角色方案的质量和成本。

本章继续使用资料研究任务。多个角色可以调用同一个模型；分工是否有效，取决于任务、信息和权限是否清楚。

[返回课程入口](../README.md) · 理解基本循环后再看。


## 1. 常见角色分别做什么

| 角色 | 工作 | 典型输出 |
| --- | --- | --- |
| planner | 拆分任务，安排依赖 | 子问题和执行计划 |
| executor / researcher | 执行子任务，读取资料 | 原文、来源和缺失信息 |
| writer | 根据证据组织答案 | 带引用的稿件 |
| reviewer | 检查稿件是否符合原题和证据 | 接受，或列出具体问题 |
| critic | 寻找反例和薄弱假设 | 需要核查的风险或缺陷 |
| router | 把请求交给合适角色或工具 | 下一处理者 |

不需要把所有角色都用上。一个简单问题可能一次回答就够；给同一个模型换几个角色名称，也不保证产生有效分工。


## 2. 用固定流程管理交接

本章采用下面的流程：

```text
研究 → 写作 → 审核 → 接受后结束
             ↑      │
             └─需要修改时返回
```

没有证据时说明无法完成；达到修订上限时保留最后稿件和未解决问题。

这种流程由程序维护。也可以增加 supervisor，由它选择下一角色，但仍需检查跳转是否允许、预算是否足够。本章的固定流程没有展示模型自主设计团队结构。


## 3. 交接时应该传什么

写作者需要原问题、证据和最近的修改要求；审核者需要原问题、同一份证据和稿件。没有必要把所有历史网页、日志和其他任务消息都复制给每个角色。

本地实验使用较小的交接格式：

- 证据：`[{"id": "L1", "text": "蓝舟项目位于苏州，有12位成员。"}]`。
- 稿件：有长度限制的文本。
- 审核：`accepted` 布尔值、`issues` 字符串列表和 `task_id`。

`task_id` 标识当前任务，避免交接时换了目标。`accepted="yes"` 不符合布尔要求，不能自动当作通过。更完整的研究任务还应保留来源位置、缺失信息和冲突。


## 4. 一次修订可以怎样发生

下面是教学示例：

1. 写作者输出“蓝舟项目位于苏州，有12位成员”，没有引用。
2. 审核者指出“两项事实缺少 L1 引用”。
3. 程序把原问题、证据和这条意见交回写作者。
4. 写作者补上 `[L1]`，再次审核。

要判断审核是否有用，需要看到具体意见进入下一轮，并且修订真正解决问题。仅有最后稿件，无法证明是审核改善了它。


## 5. 实际运行：本次没有发生修订

2026-10-20，CPU 上的 Qwen3-0.6B 收到固定虚构资料“蓝舟项目位于苏州，有12位成员”。

写作者首稿是“蓝舟项目位于苏州[L1]，有12位成员[L1]。”审核者接受，问题列表为空，修订次数为 0。单角色基线也正确回答并引用。

| 记录 | 写作者加审核者 | 单角色 |
| --- | --- | --- |
| 模型调用 | 2 次 | 1 次 |
| 输入 token 合计 | 239 | 67 |
| 输出 token 合计 | 39 | 17 |
| 模型请求耗时合计 | 3729.075 ms | 1520.605 ms |

这些是该次请求记录，不是严格性能基准。这个简单任务没有显示额外质量收益，多角色增加了输入、调用和等待；也没有发生先前教学示例中的修订。


## 6. 出现问题时如何停止

| 问题 | 程序处理 |
| --- | --- |
| 审核一直不接受 | 达到修订上限，报告未解决意见 |
| 任务 ID 变了 | 返回任务漂移，不继续执行新目标 |
| 审核字段或类型错误 | 返回无效审核 |
| 证据为空或格式错误 | 在生成前拒绝交接 |
| 每轮输入持续变长 | 只传必要证据和最近意见，并保留原文以便回读 |

固定异常测试已覆盖这些控制分支，但不能证明真实模型能抵御所有漂移或恶意输入。按字符截断证据也可能丢失重要条件，需要注意实际保留了什么。


## 7. 什么时候先用单 Agent

资料少、问题明确、一次回答就能检查时，先用单 Agent。任务可以独立研究、需要不同工具范围或确有独立审阅需求时，再考虑多个角色。

要判断角色隔离是否改善质量，需要相同任务、资料和评分，并比较总调用、token 和时间。还可以加入相同总预算下的单 Agent 自检组。本页的一次生成与两角色对比尚未完成这种严格比较。

下一页：[观察真实角色交接](02-live-coordination.ipynb)。


## 代码说明

`WritingState` 保存流程状态，`researcher` 提供教学证据，`coordinate` 管理写作、审核和修订。默认写作者及审核者是固定函数，用来检查循环和终止行为，不会调用模型。


In [ ]:
from miniagent_course import ROOT, OUTPUT, load_lesson


In [ ]:
from __future__ import annotations
import copy, json, math, re, time, tempfile, os, sys
from dataclasses import dataclass
from pathlib import Path
from urllib.request import Request, build_opener, ProxyHandler
os.environ["PYTHONIOENCODING"] = "utf-8"

In [ ]:
runtime = load_lesson('03-agent-harness/01-harness.ipynb')
schema = runtime.schema
validate = runtime.validate
Tool = runtime.Tool
calculator = runtime.calculator
calculator_tool = runtime.calculator_tool
call = runtime.call
final = runtime.final
ReplayModel = runtime.ReplayModel
normalize_tool_message = runtime.normalize_tool_message
OllamaModel = runtime.OllamaModel
chunks = runtime.chunks
terms = runtime.terms
DocumentIndex = runtime.DocumentIndex
citation_errors = runtime.citation_errors
SessionStore = runtime.SessionStore
Harness = runtime.Harness
compact_completed_turns = runtime.compact_completed_turns

In [ ]:
from dataclasses import dataclass, field
@dataclass
class WritingState:
    topic: str
    evidence: list = field(default_factory=list)
    draft: str = ""
    issues: list = field(default_factory=list)
    revision: int = 0
    status: str = "research"

def researcher(topic):
    return DocumentIndex({"notes.md": "Agent 使用工具并依据反馈调整下一步。"}).search(topic)

def reviewer(draft, evidence):
    issues = citation_errors(draft, evidence)
    if len(draft) > 300:
        issues.append("too_long")
    return {"accepted": not issues, "issues": issues}

In [ ]:
def coordinate(topic, writer, max_revisions=2):
    state, events = WritingState(topic), []
    state.evidence = researcher(topic)
    events.append({"role": "research", "evidence": state.evidence})
    if not state.evidence:
        state.status = "no_evidence"
        return state, events
    for revision in range(max_revisions + 1):
        state.revision = revision
        state.draft = writer(state.topic, state.evidence, list(state.issues))
        checked = reviewer(state.draft, state.evidence)
        state.issues = checked["issues"]
        events.append({"role": "review", "revision": revision, **checked})
        if checked["accepted"]:
            state.status = "completed"
            return state, events
    state.status = "revision_limit"
    return state, events

def fixture_writer(topic, evidence, issues):
    return "Agent 使用工具并依据反馈调整下一步。" + (" [L1]" if issues else "")
state, events = coordinate("Agent", fixture_writer)
print(state)
print(events)
assert state.status == "completed" and state.revision == 1
assert coordinate("不存在的资料", fixture_writer)[0].status == "no_evidence"
assert coordinate("Agent", lambda *args: "无引用")[0].status == "revision_limit"

In [ ]:
evidence = researcher("Agent")
single_draft = fixture_writer("Agent", evidence, ["请加引用"])
print({"single_writer_calls": 1, "coordinated_writer_calls": state.revision+1,
       "same_answer": single_draft == state.draft})